In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/drive/MyDrive/colab/data/adm/first-iteration/

In [ ]:
!wget -c -O /content/drive/MyDrive/colab/data/adm/first-iteration/glami_images.tar.gz "https://filesender.cesnet.cz/download.php?token=cbe27425-aa43-480a-982d-c5ab14179a91&archive_format=undefined&files_ids=829843"

--2026-03-30 13:53:19--  https://filesender.cesnet.cz/download.php?token=cbe27425-aa43-480a-982d-c5ab14179a91&archive_format=undefined&files_ids=829843
Resolving filesender.cesnet.cz (filesender.cesnet.cz)... 78.128.212.102, 2001:718:ff03:400::1:102
Connecting to filesender.cesnet.cz (filesender.cesnet.cz)|78.128.212.102|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12208583091 (11G) [application/x-gzip]
Saving to: ‘/content/drive/MyDrive/colab/data/adm/first-iteration/glami_images.tar.gz’

/content/drive/MyDr 100%[===================>]  11.37G  24.7MB/s    in 8m 40s  

2026-03-30 14:01:59 (22.4 MB/s) - ‘/content/drive/MyDrive/colab/data/adm/first-iteration/glami_images.tar.gz’ saved [12208583091/12208583091]



First, let's create the target directory `/content/images` if it doesn't already exist. This is where the contents of the archive will be extracted.

In [2]:
import os

extraction_dir = '/content/images'
os.makedirs(extraction_dir, exist_ok=True)
print(f"Directory '{extraction_dir}' ensured.")

Directory '/content/images' ensured.


Now, let's copy the `glami_images.tar.gz` file from Google Drive to the `/content` directory. Copying it locally first can sometimes speed up the extraction process, especially if dealing with many small files, as it avoids repeated access over the network to Google Drive during extraction.

In [3]:
import shutil
import os

source_archive_path = '/content/drive/MyDrive/colab/data/adm/first-iteration/glami_images.tar.gz'
destination_archive_path = '/content/glami_images.tar.gz'

print(f"Copying {source_archive_path} to {destination_archive_path}...")
shutil.copyfile(source_archive_path, destination_archive_path)
print("Copy complete.")

Copying /content/drive/MyDrive/colab/data/adm/first-iteration/glami_images.tar.gz to /content/glami_images.tar.gz...


KeyboardInterrupt: 

Finally, we will extract the copied `glami_images.tar.gz` file into the `/content/images` directory. This might take some time as the archive is quite large (12GB).

In [ ]:
import tarfile
import os
from tqdm.auto import tqdm

archive_path = '/content/glami_images.tar.gz'
extraction_dir = '/content/images'

print(f"Extracting {archive_path} to {extraction_dir}...")

with tarfile.open(archive_path, "r:gz") as tar:
    # Get all members for progress bar
    members = tar.getmembers()
    for member in tqdm(members, desc="Extracting files", unit="file"):
        try:
            tar.extract(member, path=extraction_dir)
        except Exception as e:
            print(f"Error extracting {member.name}: {e}")

print("Extraction complete.")

# Verify extraction by listing contents and checking size
print("\nVerifying extracted content:")
!ls {extraction_dir} | head -n 5
!du -sh {extraction_dir}

First, let's remove any `.jpg` files that might have been incorrectly extracted directly into `/content/drive/MyDrive`.

Now that any misplaced `.jpg` files are removed, please re-run the extraction cell (cell `017ff3f8`). It has been modified to first remove the existing `fit_dataset_images` directory and then perform a fresh, complete extraction of the 12GB `glami_images.tar.gz` archive to `/content/drive/MyDrive/colab/data/adm/first-iteration/`.

In [ ]:
import pandas as pd

It's indeed suspicious that the extracted directory is only 5.9MB while the archive is 12GB. Let's inspect the contents of the `glami_images.tar.gz` file to understand its internal structure and identify where the 12GB of data is located.

In [ ]:
import tarfile
import os

archive_path = '/content/drive/MyDrive/colab/data/adm/first-iteration/glami_images.tar.gz'

print(f"Listing contents of {archive_path}:")

# Open the tar.gz file in read mode
with tarfile.open(archive_path, "r:gz") as tar:
    # Get all members (files and directories) in the archive
    members = tar.getmembers()

    # Print the names and sizes of the first 20 members to get an idea of the structure
    print("First 20 members:")
    for i, member in enumerate(members):
        if i >= 20:
            break
        print(f"  {member.name} (size: {member.size / (1024*1024):.2f} MB)")

    # Optionally, count total number of files/directories and total uncompressed size
    total_files = len(members)
    total_uncompressed_size = sum(member.size for member in members)
    print(f"\nTotal members in archive: {total_files}")
    print(f"Total uncompressed size: {total_uncompressed_size / (1024*1024*1024):.2f} GB")

    # Check for the presence of the 'fit_dataset_images' directory at the root of the archive
    # This assumes 'fit_dataset_images' is a direct child of the archive's root.
    fit_dataset_images_present = any(m.name.startswith('fit_dataset_images/') or m.name == 'fit_dataset_images' for m in members)
    print(f"'fit_dataset_images' related content present in archive: {fit_dataset_images_present}")

    # If 'fit_dataset_images' is present, try to find its size within the tarfile
    fit_dataset_images_total_size = 0
    for member in members:
        if member.name.startswith('fit_dataset_images/') or member.name == 'fit_dataset_images':
            fit_dataset_images_total_size += member.size
    if fit_dataset_images_total_size > 0:
        print(f"Total size of 'fit_dataset_images' content within archive: {fit_dataset_images_total_size / (1024*1024*1024):.2f} GB")
    else:
        print("Could not determine total size for 'fit_dataset_images' content within archive (it might not be there or is empty).")


Listing contents of /content/drive/MyDrive/colab/data/adm/first-iteration/glami_images.tar.gz:
First 20 members:
  ._fit_dataset_images (size: 0.00 MB)
  fit_dataset_images (size: 0.00 MB)
  fit_dataset_images/._1007645.jpg (size: 0.00 MB)
  fit_dataset_images/1007645.jpg (size: 0.02 MB)
  fit_dataset_images/._985068.jpg (size: 0.00 MB)
  fit_dataset_images/985068.jpg (size: 0.01 MB)
  fit_dataset_images/._534028.jpg (size: 0.00 MB)
  fit_dataset_images/534028.jpg (size: 0.01 MB)
  fit_dataset_images/._1191633.jpg (size: 0.00 MB)
  fit_dataset_images/1191633.jpg (size: 0.02 MB)
  fit_dataset_images/._1260672.jpg (size: 0.00 MB)
  fit_dataset_images/1260672.jpg (size: 0.01 MB)
  fit_dataset_images/._329544.jpg (size: 0.00 MB)
  fit_dataset_images/329544.jpg (size: 0.01 MB)
  fit_dataset_images/._1039973.jpg (size: 0.00 MB)
  fit_dataset_images/1039973.jpg (size: 0.01 MB)
  fit_dataset_images/._428373.jpg (size: 0.00 MB)
  fit_dataset_images/428373.jpg (size: 0.01 MB)
  fit_dataset_image

In [ ]:
import pandas as pd

items_train = pd.read_csv("/content/drive/MyDrive/colab/data/adm/first-iteration/items_train.csv")
items_phase_1 = pd.read_csv("/content/drive/MyDrive/colab/data/adm/first-iteration/items_phase_1.csv")
task_1 = pd.read_csv("/content/drive/MyDrive/colab/data/adm/first-iteration/task_1.csv")

In [ ]:
items_train.head()

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo,label
0,692210,148.0,"230,232",['11'],NaN,Раница Rains 14480 Черен,NaN,bg,41599
1,943360,148.0,"230,232",['11'],NaN,Раница Rains,Rains Раница 14480 Черен,bg,41599
2,447879,179.9,230,['1'],NaN,Раница Rains Trail Cord Rolltop Backpack W3 в ...,Раница от колекция Rains. Моделът е направен о...,bg,41599
3,1169543,179.9,230,"['1', '11']",NaN,Раница Rains Trail Cord Rolltop Backpack W3 в ...,- Непромокаем модел.\n- Повишена водоустойчиво...,bg,41599
4,671883,1910.0,"230,232",['11'],NaN,Batoh Rains,Rains Batoh 14480 Černá,cz,41599


In [ ]:
items_phase_1.head()

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo
0,451136,69.90,"771,772",['11'],NaN,Δερμάτινη θήκη για κάρτες Tommy Hilfiger χρώμα...,Πορτοφόλι της κολεξιόν Tommy Hilfiger. Μοντέλο...,gr
1,281651,29.95,775,['1'],NaN,Cepure Billabong,Billabong Cepure Alta Rib EBJHA00115 Rozā,lv
2,206147,76.99,230,['1'],NaN,Čizme Badura,Badura Čizme FUNCHAL-2603 Crna,hr
3,716396,69.95,6449,['42'],NaN,Sandále Froddo,Sandále Froddo Keko G3150287-10 M Modrá,sk
4,55666,29.90,"232,780",['11'],NaN,Secrid - Novčanik,- RFID Protect - zaštita od elektroničke krađe...,hr


In [ ]:
task_1.head()

,item1,item2,item3,item4,item5
0,130622,292253,463442,483968,1253745
1,82627,388496,553738,638400,884327
2,46130,333489,644448,848154,1178149
3,150796,248537,742067,1206230,1280786
4,76610,196894,345145,620255,932761


In [ ]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
import numpy as np

# Helper function to parse departmentIds (strings like "['11']")
def parse_department_ids(ids_str):
    if pd.isna(ids_str):
        return []
    try:
        # ast.literal_eval safely evaluates string representations of Python literals
        # It expects actual Python literal syntax, e.g., "['11']" or "[11]"
        return [str(x) for x in ast.literal_eval(ids_str)] # Ensure elements are strings
    except (ValueError, SyntaxError):
        # If it's not a valid list string, treat it as a single item if possible
        cleaned_str = ids_str.strip("[]'").strip()
        return [cleaned_str] if cleaned_str else []

# Helper function to parse colorTagIdsString (strings like "230,232")
def parse_color_tag_ids(ids_str):
    if pd.isna(ids_str):
        return []
    return [x.strip() for x in str(ids_str).split(',') if x.strip()] # Split by comma and strip whitespace

# Apply parsing functions to both dataframes
items_train['parsed_departmentIds'] = items_train['departmentIds'].apply(parse_department_ids)
items_phase_1['parsed_departmentIds'] = items_phase_1['departmentIds'].apply(parse_department_ids)

items_train['parsed_colorTagIdsString'] = items_train['colorTagIdsString'].apply(parse_color_tag_ids)
items_phase_1['parsed_colorTagIdsString'] = items_phase_1['colorTagIdsString'].apply(parse_color_tag_ids)

# --- One-Hot Encoding for departmentIds ---
# Collect all unique department IDs across both dataframes for consistent encoding
all_department_ids = set()
for ids_list in items_train['parsed_departmentIds']:
    all_department_ids.update(ids_list)
for ids_list in items_phase_1['parsed_departmentIds']:
    all_department_ids.update(ids_list)

mlb_dept = MultiLabelBinarizer()
# Fit MultiLabelBinarizer with all unique classes found
# We pass a list containing a list of all unique IDs to fit all labels present
mlb_dept.fit([list(all_department_ids)])

dept_ohe_train = mlb_dept.transform(items_train['parsed_departmentIds'])
dept_ohe_phase_1 = mlb_dept.transform(items_phase_1['parsed_departmentIds'])

# Create DataFrames for one-hot encoded department IDs with meaningful column names
dept_ohe_df_train = pd.DataFrame(dept_ohe_train, columns=[f'departmentId_{c}' for c in mlb_dept.classes_], index=items_train.index)
dept_ohe_df_phase_1 = pd.DataFrame(dept_ohe_phase_1, columns=[f'departmentId_{c}' for c in mlb_dept.classes_], index=items_phase_1.index)

# Concatenate one-hot encoded features with original dataframes
items_train = pd.concat([items_train, dept_ohe_df_train], axis=1)
items_phase_1 = pd.concat([items_phase_1, dept_ohe_df_phase_1], axis=1)


# --- One-Hot Encoding for colorTagIdsString ---
# Collect all unique color tag IDs across both dataframes
all_color_tag_ids = set()
for ids_list in items_train['parsed_colorTagIdsString']:
    all_color_tag_ids.update(ids_list)
for ids_list in items_phase_1['parsed_colorTagIdsString']:
    all_color_tag_ids.update(ids_list)

mlb_color = MultiLabelBinarizer()
mlb_color.fit([list(all_color_tag_ids)])

color_ohe_train = mlb_color.transform(items_train['parsed_colorTagIdsString'])
color_ohe_phase_1 = mlb_color.transform(items_phase_1['parsed_colorTagIdsString'])

# Create DataFrames for one-hot encoded color tag IDs
color_ohe_df_train = pd.DataFrame(color_ohe_train, columns=[f'colorTagId_{c}' for c in mlb_color.classes_], index=items_train.index)
color_ohe_df_phase_1 = pd.DataFrame(color_ohe_phase_1, columns=[f'colorTagId_{c}' for c in mlb_color.classes_], index=items_phase_1.index)

# Concatenate with original dataframes
items_train = pd.concat([items_train, color_ohe_df_train], axis=1)
items_phase_1 = pd.concat([items_phase_1, color_ohe_df_phase_1], axis=1)


# --- One-Hot Encoding for geo ---
# Collect all unique geo values across both dataframes
all_geo_values = pd.concat([items_train['geo'], items_phase_1['geo']]).unique().reshape(-1, 1)

ohe_geo = OneHotEncoder(handle_unknown='ignore', sparse_output=False) # Use sparse_output=False for dense array
ohe_geo.fit(all_geo_values)

geo_ohe_train = ohe_geo.transform(items_train['geo'].values.reshape(-1, 1))
geo_ohe_phase_1 = ohe_geo.transform(items_phase_1['geo'].values.reshape(-1, 1))

# Create DataFrames for one-hot encoded geo with meaningful column names
geo_ohe_df_train = pd.DataFrame(geo_ohe_train, columns=ohe_geo.get_feature_names_out(['geo']), index=items_train.index)
geo_ohe_df_phase_1 = pd.DataFrame(geo_ohe_phase_1, columns=ohe_geo.get_feature_names_out(['geo']), index=items_phase_1.index)

# Concatenate with original dataframes
items_train = pd.concat([items_train, geo_ohe_df_train], axis=1)
items_phase_1 = pd.concat([items_phase_1, geo_ohe_df_phase_1], axis=1)

# Display the head of the modified dataframes to show the new columns
print("items_train after one-hot encoding:")
display(items_train.head())
print("\nitems_phase_1 after one-hot encoding:")
display(items_phase_1.head())

# Drop intermediate parsed columns to keep the DataFrame clean
items_train = items_train.drop(columns=['parsed_departmentIds', 'parsed_colorTagIdsString'])
items_phase_1 = items_phase_1.drop(columns=['parsed_departmentIds', 'parsed_colorTagIdsString'])

items_train after one-hot encoding:


,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo,label,parsed_departmentIds,...,geo_gr,geo_hr,geo_hu,geo_it,geo_lt,geo_lv,geo_pl,geo_ro,geo_si,geo_sk
0,692210,148.0,"230,232",['11'],NaN,Раница Rains 14480 Черен,NaN,bg,41599,[11],...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,943360,148.0,"230,232",['11'],NaN,Раница Rains,Rains Раница 14480 Черен,bg,41599,[11],...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,447879,179.9,230,['1'],NaN,Раница Rains Trail Cord Rolltop Backpack W3 в ...,Раница от колекция Rains. Моделът е направен о...,bg,41599,[1],...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1169543,179.9,230,"['1', '11']",NaN,Раница Rains Trail Cord Rolltop Backpack W3 в ...,- Непромокаем модел.\n- Повишена водоустойчиво...,bg,41599,"[1, 11]",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,671883,1910.0,"230,232",['11'],NaN,Batoh Rains,Rains Batoh 14480 Černá,cz,41599,[11],...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



items_phase_1 after one-hot encoding:


,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo,parsed_departmentIds,parsed_colorTagIdsString,...,geo_gr,geo_hr,geo_hu,geo_it,geo_lt,geo_lv,geo_pl,geo_ro,geo_si,geo_sk
0,451136,69.90,"771,772",['11'],NaN,Δερμάτινη θήκη για κάρτες Tommy Hilfiger χρώμα...,Πορτοφόλι της κολεξιόν Tommy Hilfiger. Μοντέλο...,gr,[11],"[771, 772]",...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,281651,29.95,775,['1'],NaN,Cepure Billabong,Billabong Cepure Alta Rib EBJHA00115 Rozā,lv,[1],[775],...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,206147,76.99,230,['1'],NaN,Čizme Badura,Badura Čizme FUNCHAL-2603 Crna,hr,[1],[230],...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,716396,69.95,6449,['42'],NaN,Sandále Froddo,Sandále Froddo Keko G3150287-10 M Modrá,sk,[42],[6449],...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,55666,29.90,"232,780",['11'],NaN,Secrid - Novčanik,- RFID Protect - zaštita od elektroničke krađe...,hr,[11],"[232, 780]",...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# check if all items have images


In [ ]:
from torch.utils.data import Dataset
import os
from PIL import Image

class GlamiDataset(Dataset):

    def __init__(
            self,
            dataframe,
            base_image_path,
            image_resize: tuple(int, int)
    ):
        self.data = dataframe
        self.base_image_path = base_image_path

        self.image_transform = transforms.Compose([
            transforms.Resize(image_resize),
            transforms.ToTensor(),
        ])

    def __load_image(self, label_id):
        image_path = os.path.join(self.base_image_path, f"{label_id}.jpg")
        image = Image.open(image_path)
        image = image.convert("RGB")
        image = self.image_transform(image)
        return image

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data.iloc[idx]
        image = self.__load_image(item["id"])
        metadata = item.drop(["id"])
        label = item["label"]
        text_descr = item["description"]
        text_title = item["title"]
        text = """TITLE: {text_title}

        DESCRIPTION: {text_descr}
        """.format(text_title=text_title, text_descr=text_descr)
        return image, text, metadata, label


# Models

In [ ]:
from torch import nn
import torch

class ItemEncoder(nn.Module):

    def __init__(
            self,
            image_encoder,
            text_encoder,
            input_size,
            mlp_layer_sizes: list[int],
            mlp_activation = nn.Tanh,
        ):
        self.image_encoder = image_encoder
        self.text_encoder = text_encoder

        # MLP
        mlp_layers = []
        prev_layer_size = input_size
        for layer_size in mlp_layer_sizes:
            mlp_layers.append(nn.Linear(input_size, layer_size))
            mlp_layers.append(mlp_activation())
            prev_layer_size = layer_size

        self.mlp = nn.Sequential(*mlp_layers)

    def forward(
            self,
            image,  # (batch_size, image_size)
            text,  # (batch_size, 1)
            metadata  # (batch_size, metadata_size)
        ):
        image_emb = self.image_encoder(image)  # (batch_size, embedding_size)
        text_emb = self.text_encoder(text)  # (batch_size, embedding_size)
        input = torch.cat(
            [image_emb, text_emb, metadata],
            dim=1
        )  # (batch_size, embedding_size + metadata_size)
        output = self.mlp(input)  # (batch_size, output_size)
        return output

# Training

Using contrative loss

## Loss

In [ ]:
def info_nce_loss(features, labels, temperature=0.5):
    device = features.device
    batch_size = features.shape[0]
    labels = labels.contiguous().view(-1, 1)
    mask = torch.eq(labels, labels.T).float().to(device)
    contrast_count = features.shape[1]
    contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
    anchor_feature = contrast_feature
    anchor_count = contrast_count
    anchor_dot_contrast = torch.div(
        torch.matmul(anchor_feature, contrast_feature.T),
        temperature
    )
    logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
    logits = anchor_dot_contrast - logits_max.detach()
    mask = mask.repeat(anchor_count, contrast_count)
    logits_mask = torch.scatter(
        torch.ones_like(mask),
        1,
        torch.arange(batch_size * anchor_count).view(-1, 1).to(device),
        0
    )
    mask = mask * logits_mask
    exp_logits = torch.exp(logits) * logits_mask
    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))
    mean_log_prob_pos = (mask * log_prob).sum(1) / mask.sum(1)
    loss = - (temperature / batch_size) * mean_log_prob_pos
    loss = loss.view(anchor_count, batch_size).mean()
    return loss

## Training Loop

In [ ]:
epochs = 3
batch_size = 5
learning_rate = 1e-3

for epoch in range(epochs):
    for image, text, metadata, label in train_loader: